## Date Dimension Table - PySpark Cleaning

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, to_date, lit
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("DateDimCleaning").getOrCreate()

df_date = spark.read.csv("../Coursework_dataset/date_dim_helper.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 18:20:07 WARN Utils: Your hostname, saag, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface wlp1s0)
26/04/14 18:20:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 18:20:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 18:20:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 18:20:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


### Preview the date dimension table

In [2]:
df_date.show(5)

+-------+----------+-----------+----------+------------+----------+-------+----+----------+-------------+
|date_id| full_date|day_of_week|day_number|month_number|month_name|quarter|year|is_weekend|is_np_holiday|
+-------+----------+-----------+----------+------------+----------+-------+----+----------+-------------+
|      1|2019-01-01|    Tuesday|         1|           1|   January|     Q1|2019|         0|            0|
|      2|2019-01-02|  Wednesday|         2|           1|   January|     Q1|2019|         0|            0|
|      3|2019-01-03|   Thursday|         3|           1|   January|     Q1|2019|         0|            0|
|      4|2019-01-04|     Friday|         4|           1|   January|     Q1|2019|         0|            0|
|      5|2019-01-05|   Saturday|         5|           1|   January|     Q1|2019|         1|            0|
+-------+----------+-----------+----------+------------+----------+-------+----+----------+-------------+
only showing top 5 rows


### Check how many values each column has (non-null count)

In [3]:
non_null_counts = df_date.select([
    count(when(col(c).isNotNull(), c)).alias(c) for c in df_date.columns
])
non_null_df = non_null_counts.toPandas().T.reset_index()
non_null_df.columns = ['Column Name', 'Non-Null Count']
non_null_df

,Column Name,Non-Null Count
0,date_id,2557
1,full_date,2557
2,day_of_week,2557
3,day_number,2557
4,month_number,2557
5,month_name,2557
6,quarter,2557
7,year,2557
8,is_weekend,2557
9,is_np_holiday,2557


### Check which columns have null values

In [4]:
null_counts = df_date.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_date.columns
])
null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['Column Name', 'Null Count']
null_df

,Column Name,Null Count
0,date_id,0
1,full_date,0
2,day_of_week,0
3,day_number,0
4,month_number,0
5,month_name,0
6,quarter,0
7,year,0
8,is_weekend,0
9,is_np_holiday,0


### Check for duplicate date_id rows

In [5]:
duplicates = df_date.join(
    df_date.groupBy("date_id").count().filter(col("count") > 1).select("date_id"),
    on="date_id",
    how="inner"
)

print("Duplicate date_id rows:", duplicates.count())

Duplicate date_id rows: 0


### Fix 1: Convert full_date to a proper date type

In [6]:
df_date = df_date.withColumn("full_date", to_date(col("full_date")))

print("Unparseable full_date values:", df_date.filter(col("full_date").isNull()).count())

Unparseable full_date values: 0


### Fix 2: Convert quarter labels to numeric values
Quarter is stored as Q1, Q2, Q3, Q4 (string). 
We convert to 1, 2, 3, 4 (integer) so date calculations work correctly.
For example, sorting by quarter or computing quarterly aggregates becomes straightforward with numbers.

In [7]:
# Show the current quarter values before converting
df_date.select("quarter").distinct().orderBy("quarter").show()

# Map Q1 -> 1, Q2 -> 2, Q3 -> 3, Q4 -> 4 using a lookup map
quarter_mapping = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
mapping_expr = F.create_map([F.lit(x) for x in sum(quarter_mapping.items(), ())])

df_date = df_date.withColumn(
    "quarter",
    mapping_expr[col("quarter")].cast(IntegerType())
)

# Confirm the new values
df_date.select("quarter").distinct().orderBy("quarter").show()

+-------+
|quarter|
+-------+
|     Q1|
|     Q2|
|     Q3|
|     Q4|
+-------+

+-------+
|quarter|
+-------+
|      1|
|      2|
|      3|
|      4|
+-------+



### Verify distinct values per column
A quick sanity check - e.g. 7 day names, 12 months, 4 quarters, etc.

In [8]:
for c in ["day_of_week", "month_number", "month_name", "quarter", "is_weekend", "is_np_holiday"]:
    distinct_count = df_date.select(c).distinct().count()
    print(f"{c}: {distinct_count} distinct values")

day_of_week: 7 distinct values
month_number: 12 distinct values
month_name: 12 distinct values
quarter: 4 distinct values
is_weekend: 2 distinct values
is_np_holiday: 2 distinct values


### Check the schema to confirm data types

In [9]:
df_date.printSchema()

root
 |-- date_id: integer (nullable = true)
 |-- full_date: date (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- day_number: integer (nullable = true)
 |-- month_number: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_np_holiday: integer (nullable = true)



### Export the cleaned date dimension table

In [10]:
df_date.coalesce(1).write.csv("../cleaned_dataset/date_dim_cleaned.csv", header=True, mode="overwrite")
print("Date dimension table exported successfully to 'cleaned_dataset/date_dim_cleaned.csv'")

Date dimension table exported successfully to 'cleaned_dataset/date_dim_cleaned.csv'
